In [ ]:
%cd ..

# Linear Regression Baseline — Store Sales

Illustrative walkthrough of the **log-Ridge** baseline with **recursive
multi-step forecasting** — the same pipeline used by
`scripts/train_linear.py`.

Recursive prediction feeds each day's forecast back as the lag feature for
the next day, mirroring real test-time conditions (where actuals for the
16-day horizon are unknown).  This avoids the stale-lag collapse that plain
direct prediction produces.

This notebook trains on a **5-store smoke subsample** so it runs in seconds.
For the full dataset:

```bash
make train-linear CONFIG=config/linear-log.yaml RUN_NAME=my-log-ridge
```

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge

from store_sales.data import load_config, load_data, merge_tables, timeseries_split
from store_sales.features import TimeSeriesFeatureEngineer
from store_sales.metrics import rmsle
from store_sales.recursive import onehot_align, onehot_fit, recursive_forecast

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

CONFIG_PATH = "config/linear-smoke.yaml"
cfg = load_config(CONFIG_PATH)
print(f"Config: {CONFIG_PATH}  (run_scope={cfg.get('run_scope', '?')})")

## 1. Load & merge data

We join store metadata, oil prices, holidays, and transactions onto the train/test
frames — the same pipeline used by `scripts/train_linear.py`.

In [ ]:
tables = load_data(cfg["paths"]["data"])
train, test = merge_tables(tables)

# Smoke subsample: 5 stores
max_stores = cfg.get("subsample", {}).get("max_stores", 5)
keep_stores = sorted(train["store_nbr"].unique())[:max_stores]
train = train[train["store_nbr"].isin(keep_stores)].copy().reset_index(drop=True)
test = test[test["store_nbr"].isin(keep_stores)].copy().reset_index(drop=True)

target_col = cfg["competition"]["target"]
target_transform = cfg["model"].get("target_transform", "raw")
y_full_raw = train[target_col].copy()
if target_transform == "log1p":
    train["log_sales"] = np.log1p(train[target_col])
    test["log_sales"] = 0.0
    y_full = train["log_sales"].copy()
else:
    y_full = y_full_raw.copy()
print(f"Train: {train.shape}  Test: {test.shape}")
print(f"Stores: {sorted(train['store_nbr'].unique())}")
print(f"Date range: {train['date'].min()} → {train['date'].max()}")
print(f"target_transform: {target_transform}")

## 2. Feature engineering

**Critical rule:** lag and rolling features must be computed on the full sorted
store-family history *before* the train/validation split.

Features used:
- **Lags**: log_sales from 1, 7, 14, 28 days ago
- **Rolling**: 7-day mean of log_sales
- **Fourier**: dayofyear (3 harmonics) + dayofweek (2 harmonics) for cyclical seasonality
- **External**: onpromotion, oil price, holidays

In [ ]:
feat_cfg = cfg["features"]
engineer = TimeSeriesFeatureEngineer(
    date_col=feat_cfg.get("date_col", "date"),
    store_col=feat_cfg.get("store_col", "store_nbr"),
    family_col=feat_cfg.get("family_col", "family"),
    onpromotion_col=feat_cfg.get("onpromotion_col", "onpromotion"),
    date_features=feat_cfg.get("date_features", []),
    drop_cols=feat_cfg.get("drop_cols", []),
    lag_config=feat_cfg.get("lag_features", []),
    rolling_config=feat_cfg.get("rolling_features", []),
    fourier_config=feat_cfg.get("fourier_features", None),
    ref_date=train["date"].min(),
)

X_lag, X_test_feat = engineer.create_lag_features(train, test, target_col)
engineer.fit(X_lag)
print(f"Feature matrix (pre-split): {X_lag.shape}")
print(f"Sample columns: {list(X_lag.columns[:8])} ...")

## 3. Train / validation split + one-hot encoding

Hold out the last 16 days as validation — mimics the competition's 16-day test horizon.

Categorical columns (store_nbr, family, type) are one-hot encoded with
`onehot_fit` on the training split, then val/test are aligned to the same
column space via `onehot_align`.

In [ ]:
val_days = cfg.get("timeseries", {}).get("test_period_days", 16)
X_train_raw, X_val_raw = timeseries_split(X_lag, val_days)
y_train = y_full.loc[X_train_raw.index]
y_val_raw = y_full_raw.loc[X_val_raw.index]

X_train = engineer.transform(X_train_raw)
X_train, known_cats, feature_names = onehot_fit(X_train)

val_split_date = sorted(train["date"].unique())[-val_days]
print(f"Train rows: {len(X_train)}  Val rows: {len(X_val_raw)}  Features: {len(feature_names)}")
print(f"Validation starts: {val_split_date}")

## 4. Fit Ridge + recursive validation

A single global Ridge model across all store-family series.  Validation uses
**recursive multi-step forecasting**: each val day's prediction is fed back
as the lag for the next day, mirroring test-time conditions.

In [ ]:
alpha = cfg["model"].get("alpha", 1.0)
ridge = Ridge(alpha=alpha, fit_intercept=True, random_state=42)
ridge.fit(X_train, y_train)

lag_target_col = engineer.lag_config[0][0] if engineer.lag_config else target_col
val_preds = recursive_forecast(
    ridge, engineer, X_train_raw, X_val_raw,
    lag_target_col=lag_target_col,
    target_transform=target_transform,
    known_cats=known_cats,
    feature_names=feature_names,
)
val_score = rmsle(y_val_raw.values, val_preds)
print(f"Validation RMSLE (recursive): {val_score:.4f}")

## 5. Recursive test forecast

Generate the 16-day test submission using the same recursive approach.
The full training history (including the val period) feeds the lags.

In [ ]:
test_preds = recursive_forecast(
    ridge, engineer, X_lag, X_test_feat,
    lag_target_col=lag_target_col,
    target_transform=target_transform,
    known_cats=known_cats,
    feature_names=feature_names,
)
print(f"Test predictions: mean={test_preds.mean():.1f}  median={np.median(test_preds):.1f}")
print(f"  range: [{test_preds.min():.1f}, {test_preds.max():.1f}]")

## 6. Visualise one series

Plot actual vs predicted for the highest-volume store-family combination.
Val predictions are recursive (honest); test predictions extend the forecast.

In [ ]:
# Pick hero series (highest total sales)
totals = train.groupby(["store_nbr", "family"])["sales"].sum().sort_values(ascending=False)
hero_store, hero_family = totals.index[0]

meta = X_lag[["date", "store_nbr", "family"]].copy()
meta["actual"] = y_full_raw.loc[X_lag.index].values
# In-sample train predictions (direct) for context
X_all_ohe = onehot_align(engineer.transform(X_lag), known_cats, feature_names)
train_preds_log = ridge.predict(X_all_ohe)
train_preds = np.expm1(train_preds_log) if target_transform == "log1p" else train_preds_log
meta["predicted"] = np.maximum(train_preds, 0)

# Override val rows with recursive predictions
val_meta = X_val_raw[["date", "store_nbr", "family"]].copy()
val_meta["pred"] = val_preds
meta.loc[X_val_raw.index, "predicted"] = val_meta["pred"].values

hero = meta[(meta["store_nbr"] == hero_store) & (meta["family"] == hero_family)].sort_values("date")
val_start = pd.Timestamp(val_split_date)
hero = hero[hero["date"] >= val_start - pd.Timedelta(days=90)]

fig, ax = plt.subplots(figsize=(14, 4.5))
pre_val = hero[hero["date"] < val_start]
post_val = hero[hero["date"] >= val_start]

ax.plot(pre_val["date"], pre_val["actual"], color="#aaaaaa", linewidth=0.6, alpha=0.7, label="Train actual")
ax.plot(post_val["date"], post_val["actual"], color="#1f77b4", linewidth=1.2, label="Val actual")
ax.plot(post_val["date"], post_val["predicted"], color="#ff7f0e", linewidth=1.2, linestyle="--", label="Val predicted (recursive)")
ax.axvline(val_start, color="black", linewidth=0.8, linestyle=":", alpha=0.5)
ax.set_title(f"Store {hero_store} — {hero_family}  (val RMSLE={val_score:.4f})", fontweight="bold")
ax.set_ylabel("Sales")
ax.legend(fontsize=8)
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

## 7. Daily aggregate view

Sum predictions across all stores to see the big-picture fit during validation.

In [ ]:
daily = meta.groupby("date").agg(actual=("actual", "sum"), predicted=("predicted", "sum")).reset_index()
daily = daily[daily["date"] >= val_start - pd.Timedelta(days=60)]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily["date"], daily["actual"], color="black", linewidth=1.0, label="Actual")
ax.plot(daily["date"], daily["predicted"], color="#ff7f0e", linewidth=0.9, linestyle="--", label="Ridge predicted")
ax.axvline(val_start, color="gray", linestyle=":", alpha=0.6)
ax.set_title("Daily aggregate sales — smoke subsample", fontweight="bold")
ax.set_ylabel("Total sales")
ax.legend()
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

## Next steps

| Command | What it does |
|---|---|
| `make train-linear CONFIG=config/linear-log.yaml RUN_NAME=demo` | Full-dataset log-Ridge with recursive forecast (~18s, RMSLE ~0.51) |
| `make benchmark-linear` | Compare all linear variants side-by-side |
| `make viz-gif` | Render animated pipeline GIF for README |
| `notebooks/visualize_predictions.ipynb` | Multi-model comparison plots |